# 03 · How does CLIP know which caption goes with which photo?

Images are vectors (Notebook 01), and so is text (Notebook 02). Now the two finally meet. We'll rebuild the [CLIP paper](https://arxiv.org/abs/2103.00020)'s training objective, its **Figure 3**, line by line on a few real photos and captions, then reuse the same scores to search.

<img src="../assets/diagrams/notebook3/fig1.png" alt="CLIP paper Figure 1: contrastive pre-training builds a matrix of image-text scores" width="800">

The picture to keep in mind: every image scored against every caption, with the correct pairs on the diagonal.

**Read more in [the book](https://arxiv.org/abs/2605.07544):** Chapter 4, Section 4.2.1, *Contrastive Alignment*, and Chapter 1, Section 1.4.3, *Fusion and Alignment Mechanisms*.

> **How to use this notebook:** Run the cells top to bottom. When you hit a **Pause and predict**, take a guess before running the next cell. You'll remember way more that way. Cells marked **TODO** are yours to play with: change a value, re-run, and see what happens.

### Pick your GPU

Got a GPU (or a few)? Set `GPU_ID` in the next cell to the one you want. Run `nvidia-smi` in a terminal to see the list. Only have one? Leave it at `"0"`. No GPU at all? Set it to `""` and everything runs on the CPU. It's slower, but it works.

This cell has to run *before* PyTorch is imported, so if you change it later, restart the kernel and run from the top.

In [ ]:
# TODO: choose your GPU. This has to run before torch is imported.
#   "0"  -> the first GPU (leave this if you only have one)
#   "1"  -> the second GPU, and so on (`nvidia-smi` lists them)
#   ""   -> no GPU, run everything on the CPU
#   None -> don't restrict anything; PyTorch sees every GPU
GPU_ID = "0"

import os
if GPU_ID is not None:
    os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"  # number GPUs the same way nvidia-smi does
    os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 1. Which images and captions are we matching?

To watch CLIP compare things, we need a small batch where we *know* the right answers: a few images, each with its own caption. Let's build one.

### 1.1 Grab our tools

Most of these you've seen before. Two are new: `defaultdict` will help us collect several captions for the same image, and `fill` just wraps long captions so they fit under a plot.

In [ ]:
from collections import defaultdict  # grouping captions, wrapping plot titles, tensors, data, and CLIP
from textwrap import fill

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from datasets import load_dataset
from transformers import CLIPModel, CLIPProcessor

Both towers and both input batches need to live on the same device, or the comparison won't work. So we pick it once, right here, and use it for everything.

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # GPU if we have one, CPU otherwise

### 1.2 Where do the captions come from?

We'll use a small slice of [COCO](https://cocodataset.org/) captions. Each row has three things: an image, one caption a person wrote for it, and a COCO ID. Most images got described by several people, so the same ID shows up on several rows. That's how we'll find all the captions for one image.

We also pin a **revision**, which is just a specific saved version of the dataset. That way you get the same four examples even if someone updates the dataset later.

<img src="../assets/diagrams/notebook3/erland_dataset.png" alt="Image, caption, and COCO identifier fields" width="800">

In [ ]:
DATASET_ID = "Erland/coco_captions_small"  # a small COCO-captions dataset on the Hugging Face Hub
DATASET_REVISION = "a516b5101cec691eb65f455ffdb054afb2fdd5dd"  # pin one exact snapshot so the examples never change

Same CLIP-B/32 checkpoint as the last two notebooks. And we'll keep the batch tiny: four pairs give us a 4 × 4 score matrix. That's sixteen comparisons, few enough to look at one by one.

In [ ]:
MODEL_ID = "openai/clip-vit-base-patch32"  # one checkpoint, both towers
SAMPLE_COUNT = 4  # TODO (optional, see TODO 4 at the end): try 8, then Run All

### 1.3 Load the caption records

We load the whole split, because one image's captions are spread across separate rows. The printout is a quick check that the columns we need (`image`, `caption`, `cocoid`) are there.

In [ ]:
dataset = load_dataset(  # load every row; one image's captions are spread over several rows
    DATASET_ID,
    split="train",
    revision=DATASET_REVISION,
)
print(dataset)

The row layout hides the "one image, many captions" structure, so let's put it back. Two dictionaries do the job: one keeps a single image per COCO ID, the other collects every caption for that ID.

In [ ]:
images_by_id = {}  # COCO ID -> one image
captions_by_id = defaultdict(list)  # COCO ID -> every caption written for it

One pass through the dataset fills both. `setdefault` only stores an image the first time it sees an ID, while the caption list grows on every row.

In [ ]:
for row in dataset:
    images_by_id.setdefault(row["cocoid"], row["image"])  # keep the first copy of each image
    captions_by_id[row["cocoid"]].append(row["caption"])  # but keep every caption

### 1.4 Pick four images with at least two captions

Why at least two? The first caption goes into our CLIP batch. The second one sits on the bench until Section 5, where we combine two captions into one search query.

In [ ]:
sample_ids = [  # the first few images with two or more captions (Section 5 needs the second one)
    cocoid for cocoid, captions_for_image in captions_by_id.items()
    if len(captions_for_image) >= 2
][:SAMPLE_COUNT]
assert len(sample_ids) == SAMPLE_COUNT, f"only {len(sample_ids)} images have two captions; lower SAMPLE_COUNT"

Now the most important detail in the whole notebook, even though it looks boring. Both lists get built from the **same ordered IDs**. So image `i` and caption `i` are always a true pair, which means the correct pairs will land on the **diagonal** of every score matrix we build. Keep that in mind; everything later depends on it.

In [ ]:
images = [images_by_id[cocoid] for cocoid in sample_ids]  # same ID order for both lists, so pair i sits on the diagonal
captions = [captions_by_id[cocoid][0] for cocoid in sample_ids]  # caption 0 goes in the batch; caption 1 waits for Section 5

Let's look at our batch before anything turns into numbers. Read each panel as a pair: `I1 ↔ T1`, `I2 ↔ T2`, and so on. (`I` for image, `T` for text.)

> **Pause and predict:** Could any caption reasonably describe more than one of these images? If so, keep an eye on that pair. It might show up later as a strong score *off* the diagonal.

In [ ]:
fig, axes = plt.subplots(1, SAMPLE_COUNT, figsize=(14, 4))  # our pairs, before either side becomes a tensor
for index, (axis, image, caption) in enumerate(zip(axes, images, captions), start=1):
    axis.imshow(image)
    axis.set_title(f"I{index} ↔ T{index}\n{fill(caption, 28)}", fontsize=9)  # fill() wraps long captions
    axis.axis("off")
fig.suptitle("Four aligned records from the caption dataset")
plt.tight_layout()
plt.show()

This contact sheet is your answer key. Whenever a matrix later says "row 2, column 3", you can come back here and see which image and which caption that means. (One small thing: the plot labels count from 1, but Python indices count from 0, so `I1` is row 0.)

## 2. How do images and captions end up in the same space?

Here's the problem. CLIP has two separate towers, a **vision encoder** and a **text encoder**, and they don't even use the same internal width. You can't take a dot product between vectors of different sizes. So each tower gets its own learned **projection**, a linear layer that maps its features into one **shared space** where images and text can finally be compared.

### 2.1 Load the two-tower checkpoint

One processor handles the prep for both sides: image resizing and normalization, plus text tokenization.

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)  # the prep recipe for both images and text

Then the model, onto our device and into eval mode. We're reading the pretrained vectors, not changing any weights.

In [ ]:
model = CLIPModel.from_pretrained(MODEL_ID).to(device)  # both towers, one checkpoint
model.eval()  # we're only reading, not training: turn off dropout

Let's ask the model how wide each tower is on the inside. We already know the vision side from Notebook 01.

In [ ]:
print("vision encoder width:", model.config.vision_config.hidden_size)  # each tower's own internal width
print("text encoder width:", model.config.text_config.hidden_size)

768 for vision, 512 for text. Different, and that's fine. Neither has reached the shared space yet.

So where do they meet? Let's check the projection width.

In [ ]:
print("shared projection width:", model.config.projection_dim)  # the shared width where images and text meet

512. After projection, every image and every caption becomes one 512-number vector. (Yes, the text tower is already 512 wide. It still gets its own learned projection, because "same size" doesn't mean "same space".)

### 2.2 Prep both halves of the batch at once

One processor call can take the images *and* the captions together, and it keeps the list order. It resizes, crops, and normalizes the four images, tokenizes the four captions, and pads the captions to a common length. **Padding** just means filling the shorter captions with extra tokens so they all line up in one tensor.

In [ ]:
inputs = processor(  # one call preps all the images and all the captions, in order
    images=images,
    text=captions,
    return_tensors="pt",
    padding=True,  # pad captions to the longest one in the batch
).to(device)

The processor hands back a few tensors: `pixel_values` for the images, plus `input_ids` (the token IDs) and `attention_mask` (1 for a real token, 0 for padding) for the text.

> **Pause and predict:** What shape will `pixel_values` have? And what about `input_ids`? One of them you can work out exactly from Notebook 01. The other depends on the captions.

In [ ]:
for name, value in inputs.items():  # pixels are [B, C, H, W]; text is [B, sequence length]
    print(f"{name:16s}", tuple(value.shape))

`pixel_values` is `[4, 3, 224, 224]`: four images, three color channels, 224 × 224 pixels, same as in Notebook 01. The text tensors are `[4, L]`, where `L` is the token count of the longest caption, special start and end tokens included. Both start with a batch size of 4, and that's the part that matters: four images, four captions, still lined up.

### 2.3 Our map: the paper's Figure 3

Here's the pseudocode straight from the CLIP paper. It's written in NumPy-style code, and it's the whole training objective in nine lines of code.

<img src="../assets/diagrams/notebook3/fig3.png" alt="CLIP paper Figure 3: NumPy-like pseudocode for the core of CLIP" width="520">

The comments at the top are a legend. `I` and `T` are a minibatch of `n` aligned images and texts. `W_i` and `W_t` are the learned projections. `d_i`, `d_t`, and `d_e` are the widths: image features, text features, and the shared **e**mbedding space. And `t` is a learned temperature, which we'll get to in Section 3.

We'll number the nine code lines and rebuild them one at a time. Here's where each one shows up in this notebook:

| Line | Figure 3 | Our variable | Section |
|---|---|---|---|
| 1 | `I_f = image_encoder(I)` | `image_features` | 2.4 |
| 2 | `T_f = text_encoder(T)` | `text_features` | 2.6 |
| 3 | `I_e = l2_normalize(np.dot(I_f, W_i), axis=1)` | `image_embeddings` | 2.5 |
| 4 | `T_e = l2_normalize(np.dot(T_f, W_t), axis=1)` | `text_embeddings` | 2.7 |
| 5 | `logits = np.dot(I_e, T_e.T) * np.exp(t)` | `cosine_similarity`, then `logits` | 3.1, 3.2 |
| 6 | `labels = np.arange(n)` | `targets` | 3.3 |
| 7 | `loss_i = cross_entropy_loss(logits, labels, axis=0)` | `image_to_text_loss` | 3.4 |
| 8 | `loss_t = cross_entropy_loss(logits, labels, axis=1)` | `text_to_image_loss` | 3.5 |
| 9 | `loss = (loss_i + loss_t)/2` | `symmetric_loss` | 3.6 |

Or, as a quick sketch of what each group of lines is doing:

```text
ALIGNED INPUTS
    images I = [I0, I1, ..., I(n−1)]
    texts  T = [T0, T1, ..., T(n−1)]
                    pair i ↔ i

ENCODE EACH MODALITY
  1  I_f ← image_encoder(I)       one vision feature per image      [n, d_i]
  2  T_f ← text_encoder(T)        one text feature per caption      [n, d_t]

PROJECT AND NORMALIZE
  3  I_e ← normalize(I_f @ W_i)   image vectors in the shared space [n, d_e]
  4  T_e ← normalize(T_f @ W_t)   text vectors in the shared space  [n, d_e]

COMPARE EVERY PAIR
  5  logits ← exp(t) · I_e @ T_eᵀ                                   [n, n]
  6  labels ← [0, 1, ..., n−1]    the diagonal pair indices

LEARN IN BOTH DIRECTIONS
  7  loss_i ← cross_entropy(logits,  labels)
  8  loss_t ← cross_entropy(logitsᵀ, labels)
  9  loss   ← (loss_i + loss_t) / 2
```

> **Heads up:** We'll go slightly out of order. We finish the image side first (lines 1 and 3), then do the text side (lines 2 and 4). Same math, it just reads better one tower at a time.

### 2.4 Line 1: encode the images

> Figure 3: `I_f = image_encoder(I)  #[n, d_i]`

The vision tower reads the processed pixels and gives back one pooled feature per image. Just like in Notebook 01, that's the class token's final state (after one last layer norm). Following the paper, we'll call it `I_f` (**f** for features). It's still in the vision tower's own 768-wide world.

In [ ]:
with torch.inference_mode():  # Figure 3, line 1: I_f = image_encoder(I)
    image_output = model.vision_model(pixel_values=inputs["pixel_values"])
    image_features = image_output.pooler_output  # the class token's summary, one per image
print("I_f:", tuple(image_features.shape))  # [n, d_i]

`[4, 768]`: four images, 768 numbers each. That's `[n, d_i]`, exactly what the paper's comment says.

### 2.5 Line 3: project and normalize the images

> Figure 3: `I_e = l2_normalize(np.dot(I_f, W_i), axis=1)`

Two moves. First, `model.visual_projection` plays the role of `W_i`. It's a plain linear layer with no bias, so it really is just the matrix multiply `np.dot(I_f, W_i)`, taking 768 numbers down to 512. Second, **L2 normalization** divides each row by its length, so every image vector ends up with length 1. The paper's `axis=1` means "normalize each row", which is what `dim=-1` does on our `[4, 512]` tensor.

In [ ]:
with torch.inference_mode():  # Figure 3, line 3: I_e = l2_normalize(I_f @ W_i)
    projected_images = model.visual_projection(image_features)  # W_i: 768 -> 512
image_embeddings = F.normalize(projected_images, dim=-1)  # length 1, so dot product = cosine similarity

Let's check. We want `[4, 512]`, and every row length should be super close to 1. We need both of those for the comparison coming up.

In [ ]:
print("I_e:", tuple(image_embeddings.shape))  # [n, d_e]
print("image lengths:", image_embeddings.norm(dim=-1))  # should all be about 1

### 2.6 Line 2: encode the captions

> Figure 3: `T_f = text_encoder(T)  #[n, d_t]`

Now the text side. The text tower reads the token IDs, and we pass the attention mask so the padding tokens don't sneak into the result. Like in Notebook 02, the pooled output is the state at the end-of-text token: one feature `T_f` per caption.

In [ ]:
with torch.inference_mode():  # Figure 3, line 2: T_f = text_encoder(T)
    text_output = model.text_model(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],  # so padding can't leak into the features
    )
    text_features = text_output.pooler_output  # the end-of-text token's state, one per caption
print("T_f:", tuple(text_features.shape))  # [n, d_t]

`[4, 512]`, which is `[n, d_t]`.

### 2.7 Line 4: project and normalize the captions

> Figure 3: `T_e = l2_normalize(np.dot(T_f, W_t), axis=1)`

Same moves as line 3, now with the text projection `W_t`. The two towers can start at whatever width they like, but `I_e` and `T_e` have to finish at the same width, or we can't compare them.

In [ ]:
with torch.inference_mode():  # Figure 3, line 4: T_e = l2_normalize(T_f @ W_t)
    projected_text = model.text_projection(text_features)  # W_t: 512 -> 512
text_embeddings = F.normalize(projected_text, dim=-1)  # length 1, same as the images

Again: `[4, 512]`, row lengths about 1. And now any image row can be compared directly with any caption row.

In [ ]:
print("T_e:", tuple(text_embeddings.shape))  # [n, d_e]
print("text lengths:", text_embeddings.norm(dim=-1))  # should all be about 1 too

## 3. How does CLIP score and learn from a batch?

Lines 1 to 4 got both sides into the same space. Lines 5 to 9 are where the learning signal comes from.

### 3.1 Line 5, part one: compare every image with every caption

> Figure 3: `logits = np.dot(I_e, T_e.T) * np.exp(t)`

Let's do the first half: `I_e @ T_eᵀ`. That gives a matrix where row `i`, column `j` is the dot product between image `i` and caption `j`. And since every vector has length 1, that dot product **is** their **cosine similarity**: a score from −1 to 1 for how much two vectors point the same way. That's why we bothered normalizing.

In [ ]:
cosine_similarity = image_embeddings @ text_embeddings.T  # Figure 3, line 5 before scaling: rows = images, columns = captions
print("pairwise cosine matrix:", tuple(cosine_similarity.shape))

4 × 4, the `[n, n]` from the paper's comment. Let's see all sixteen numbers as a heatmap. The diagonal is where the true pairs live.

Don't expect a perfect diagonal, though. Captions can share objects or settings, and a pretrained model can still get things wrong, especially with only four examples to look at.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))  # rows are images, columns are captions
heatmap = ax.imshow(cosine_similarity.cpu(), cmap="viridis")
ax.set_xticks(range(SAMPLE_COUNT), [f"T{i + 1}" for i in range(SAMPLE_COUNT)])
ax.set_yticks(range(SAMPLE_COUNT), [f"I{i + 1}" for i in range(SAMPLE_COUNT)])
ax.set_xlabel("caption")
ax.set_ylabel("image")
ax.set_title("Cosine similarity before learned scaling")
fig.colorbar(heatmap, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()  # the diagonal holds the true pairs

Does the diagonal stand out? If some off-diagonal cell is bright, flip back to the contact sheet and see whether that caption could honestly fit that image too.

### 3.2 Line 5, part two: the learned temperature

Now the `* np.exp(t)` part. Cosine scores live between −1 and 1, which is a pretty narrow range. Cross-entropy (coming up in a sec) would turn such close numbers into fuzzy, unconfident probabilities. So CLIP multiplies every score by one learned number to stretch them out.

The paper calls `t` a **temperature**. CLIP actually stores its logarithm, and taking `exp` of it guarantees the multiplier is always positive. In the checkpoint, `t` lives in `model.logit_scale`. (Technically the temperature is `1 / exp(t)`. Dividing by a small temperature and multiplying by a big scale are the same thing.) The paper also caps this scale at 100 during training so it can't run off.

The scaled scores are called **logits**: raw scores that cross-entropy will turn into probabilities.

> **Pause and predict:** We're multiplying the whole matrix by one positive number. Will that change which caption scores highest for an image?

In [ ]:
logit_scale = model.logit_scale.detach().exp()  # exp(t): always positive
logits = logit_scale * cosine_similarity  # Figure 3, line 5, done

Let's print the scale and make sure the shape didn't change.

In [ ]:
print("scale:", round(logit_scale.item(), 3))  # one number for the whole matrix
print("logits:", tuple(logits.shape))  # still [n, n]

One scalar, and the matrix is still 4 × 4. And no, the winners don't change. Multiplying everything by the same positive number keeps the order within every row and column. What it changes is the *gaps* between scores, and those gaps are exactly what cross-entropy reacts to.

### 3.3 Line 6: the right answers sit on the diagonal

> Figure 3: `labels = np.arange(n)`

Remember how we built the image and caption lists from the same IDs? This is where that pays off. Image row 0's correct caption is column 0, row 1's is column 1, and so on. `arange` writes those diagonal positions as a list of class indices. (We call it `targets`; the paper calls it `labels`. Same thing.)

In [ ]:
targets = torch.arange(SAMPLE_COUNT, device=device)  # Figure 3, line 6: image i's answer is caption i
print("target caption for each image:", targets.tolist())

### 3.4 Line 7: the image-to-text loss

> Figure 3: `loss_i = cross_entropy_loss(logits, labels, axis=0)`

Here's the trick that makes CLIP work: treat each row as a tiny **classification problem**. For image `i`, the "classes" are the four captions in the batch, and the right class is caption `i`.

**Cross-entropy** is the standard classification loss. It turns a row of logits into probabilities (with softmax) and is small when the right class gets most of the probability. So it rewards a big logit on the diagonal and punishes the other captions in that row, called the **negatives**, for scoring high. `F.cross_entropy` does this for every row and averages the results.

In [ ]:
image_to_text_loss = F.cross_entropy(logits, targets)  # Figure 3, line 7: each image row picks among the captions
print("image → text loss:", round(image_to_text_loss.item(), 4))

### 3.5 Line 8: the text-to-image loss

> Figure 3: `loss_t = cross_entropy_loss(logits, labels, axis=1)`

Now flip the question. Transposing the matrix turns each caption into a row, and that row picks among the four images. The targets stay the same, because the pairing hasn't changed: caption `i` still belongs with image `i`.

> **Heads up:** The paper picks the direction with `axis=0` and `axis=1`, and it's honestly not obvious which axis means which. Don't sweat it. Line 9 averages the two, so the final loss comes out the same either way.

In [ ]:
text_to_image_loss = F.cross_entropy(logits.T, targets)  # Figure 3, line 8: transpose, so each caption row picks among the images
print("text → image loss:", round(text_to_image_loss.item(), 4))

### 3.6 Line 9: average both directions

> Figure 3: `loss = (loss_i + loss_t)/2`

Averaging gives both directions equal weight: finding the right caption for an image matters exactly as much as finding the right image for a caption. That's why people call it a **symmetric** loss. During CLIP's pretraining, the gradients of this one number update both encoders, both projections, and the temperature. Here we're just computing the value.

In [ ]:
symmetric_loss = (image_to_text_loss + text_to_image_loss) / 2  # Figure 3, line 9: average the two directions
print("symmetric loss:", round(symmetric_loss.item(), 4))

That's all nine lines. Seriously, that's CLIP's whole training objective.

**TODO:** Let's play with the temperature. Pick your own scale, recompute lines 5 to 9 with it, and see what happens to the loss. Try something tiny like `1.0` and something huge like `1000.0`.

In [ ]:
# TODO: try a tiny scale (like 1.0) and a huge one (like 1000.0)
my_scale = 1.0
my_logits = my_scale * cosine_similarity
my_loss = (F.cross_entropy(my_logits, targets) + F.cross_entropy(my_logits.T, targets)) / 2
print("my scale:", my_scale, "| symmetric loss:", round(my_loss.item(), 4))
print("same top caption for every image?", torch.equal(my_logits.argmax(dim=1), logits.argmax(dim=1)))

The loss moves around a lot, but the top caption for each image never changes. With a tiny scale, every probability is close to a quarter, so the loss sits near `ln(4)` no matter how good the vectors are. With a huge one, the model gets super confident, which is great when it's right and very costly when it's wrong.

<details>
<summary>Hint: why ln(4)?</summary>

If all four logits in a row are nearly equal, softmax gives each caption about 1/4 of the probability. Cross-entropy is −ln(probability of the right answer), so that's −ln(1/4) = ln(4), about 1.386.

</details>

### 3.7 Did we get it right? Check against the library

We rebuilt the objective by hand, but did we nail every detail? Hugging Face's `CLIPModel` can compute the same logits and loss in one forward pass. That gives us an independent answer to check against.

In [ ]:
with torch.inference_mode():  # let the library do the whole thing in one call
    reference_outputs = model(**inputs, return_loss=True)

We compare with absolute differences: one gap for the loss, and the biggest gap anywhere in the image-to-text logit matrix (`logits_per_image` in the library).

In [ ]:
loss_gap = (symmetric_loss - reference_outputs.loss).abs()  # how far apart are the two losses?
logit_gap = (logits - reference_outputs.logits_per_image).abs().max()  # biggest difference anywhere in the matrix

Both gaps should be zero, or just a tiny bit of floating-point noise. If so, we matched the library on everything: encoder outputs, projections, normalization, the learned scale, which way the matrix faces, and the averaging.

In [ ]:
print("loss gap:", loss_gap.item())  # zero or tiny floating-point noise means we matched
print("largest logit gap:", logit_gap.item())

## 4. Can the same matrix do search?

The loss is what CLIP trains on. But here's the nice bonus: that exact same score matrix also answers search questions, no extra work needed.

### 4.1 Pick the highest-scoring partner

- The max of a **row** answers "which caption fits this image best?"
- The max of a **column** answers "which image fits this caption best?"

In [ ]:
image_to_text = logits.argmax(dim=1)  # best caption for each image (row max)
text_to_image = logits.argmax(dim=0)  # best image for each caption (column max)

Compare what comes out with the targets, `[0, 1, 2, 3]`. If you see an off-diagonal index, that's not a bug in the code. It just means some other candidate beat the true partner for that query.

In [ ]:
print("image query → caption index:", image_to_text.tolist())  # compare with the targets
print("caption query → image index:", text_to_image.tolist())

### 4.2 Score the search with Recall@1

**Recall@1** asks a simple question: for what fraction of queries is the *top* result the right one? We compute it twice, once for the four image queries and once for the four caption queries.

In [ ]:
image_recall = image_to_text.eq(targets).float().mean()  # fraction of image queries whose top pick is right
text_recall = text_to_image.eq(targets).float().mean()  # same for caption queries

With four queries, every correct hit is worth `0.25`. Just don't read too much into it. Four examples are enough to show how the metric works, but nowhere near enough to say how good CLIP is on the full COCO benchmark.

In [ ]:
print("image → text R@1:", image_recall.item())  # with n queries, each hit is worth 1/n
print("text → image R@1:", text_recall.item())

## 5. What if we search with two captions at once?

Remember the second caption we left on the bench? Let's use it. Each of our images was described by at least two different people. What if we glue two descriptions together into one longer query?

### 5.1 Join two real descriptions of each image

For each image we take its first two captions, call them **A** and **B**, and join them into one string. Then we let the text transformer read the whole thing. No keyword tricks, no averaging two vectors by hand. The model figures out how to combine them.

In [ ]:
caption_pairs = [captions_by_id[cocoid][:2] for cocoid in sample_ids]  # the first two real captions for each image
composite_texts = [" ".join(pair) for pair in caption_pairs]  # glue them into one string; the text tower reads it as a whole
for index, text in enumerate(composite_texts, start=1):
    print(f"query {index}:", text)

### 5.2 Tokenize the joined queries

Same tokenizer as before. It adds the special start and end tokens, pads the batch, and **truncates** (chops off) anything longer than CLIP's text limit of 77 tokens. Two COCO captions usually fit with room to spare. Because we join the strings *before* tokenizing, attention can connect words from both descriptions.

In [ ]:
composite_inputs = processor(  # same CLIP tokenizer as before
    text=composite_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,  # chop anything past CLIP's text limit
).to(device)

Now the text tower reads each joined query in one go. That's line 2 of Figure 3 again, just with longer text. One pooled feature per query.

In [ ]:
with torch.inference_mode():  # Figure 3, line 2 again: encode each joined query in one go
    composite_output = model.text_model(**composite_inputs)
    composite_features = composite_output.pooler_output

### 5.3 Project and normalize the queries

And line 4 again: the same learned text projection takes these features into the shared space, and normalizing makes them directly comparable with the image vectors we already have.

In [ ]:
with torch.inference_mode():  # Figure 3, line 4 again: the same W_t
    composite_projected = model.text_projection(composite_features)
composite_queries = F.normalize(composite_projected, dim=-1)  # length 1, ready for cosine scores

We should see `[4, 512]` and lengths close to 1 again. Each row now stands for two descriptions read together.

In [ ]:
print("composite queries:", tuple(composite_queries.shape))  # one row per joined query
print("query lengths:", composite_queries.norm(dim=-1))  # about 1 again

### 5.4 Search our four-image catalog

Each query row gets compared with all four image vectors, and the max along the image axis is the search result. Notice we skip the temperature here. These are plain cosine scores, and since the scale never changes the winner, we don't need it for search.

> **Pause and predict:** Will two captions together find their image more reliably than one caption did back in Section 4? Write down your guess before you run the next two cells.

In [ ]:
composite_scores = composite_queries @ image_embeddings.T  # every joined query vs. every image (plain cosine, no scale)
composite_matches = composite_scores.argmax(dim=1)  # best image for each query

The indices show which image each query picked. Composite Recall@1 is the fraction of queries that found the image both captions came from. Compare it with the caption-query R@1 from Section 4.2.

In [ ]:
print("composite query → image index:", composite_matches.tolist())  # on the diagonal = found its own image
print("composite R@1:", composite_matches.eq(targets).float().mean().item())

Whatever you got, keep it in proportion. This is a tiny experiment. It shows how to build a query and search a matrix; it doesn't prove that longer queries always search better.

The heatmap shows every score, not just the winners. Rows are the joined queries (`A1+B1` means image 1's captions A and B), columns are the candidate images. A bright diagonal means each query stayed closest to its own image. Bright cells elsewhere are competitors worth checking against the contact sheet.

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 4.5))  # rows: joined queries, columns: candidate images
heatmap = ax.imshow(composite_scores.cpu(), cmap="magma")
ax.set_xticks(range(SAMPLE_COUNT), [f"I{i + 1}" for i in range(SAMPLE_COUNT)])
ax.set_yticks(range(SAMPLE_COUNT), [f"A{i + 1}+B{i + 1}" for i in range(SAMPLE_COUNT)])
ax.set_xlabel("candidate image")
ax.set_ylabel("composite text query")
ax.set_title("Two descriptions retrieve one image")
fig.colorbar(heatmap, ax=ax, label="cosine similarity")
plt.tight_layout()
plt.show()  # a bright diagonal means each query found its own image

## Recap

That's the whole trip. Both towers meet in one matrix, and that matrix does double duty:

```text
image ─→ vision encoder ─→ visual projection ─→ unit image vector ─┐   lines 1, 3
                                                                    ├─→ all pairwise logits   line 5
text  ─→ text encoder   ─→ text projection   ─→ unit text vector  ─┘   lines 2, 4
                                                                       ↓
                                              image→text and text→image losses   lines 6–9
```

**The big ideas:**

- Each tower has its own width, so each gets a learned **projection** into one shared space.
- After L2 normalization, a plain dot product **is** cosine similarity, and one matrix multiply scores every image against every caption.
- Because the batch is aligned, the **diagonal** holds the right answers and every other cell is a free **negative**.
- CLIP's loss is just cross-entropy run twice, over rows and over columns, then averaged.
- The learned temperature sharpens the scores but never changes which one wins.
- At inference time, the max of a row or column of that same matrix **is** search.

### Check your understanding

1. Why do the correct pairs land on the diagonal? What would happen to line 6 if we'd shuffled the caption list after building it?
2. Say the batch had 32 pairs instead of 4. What shape would `logits` be, and how many negatives would each image get in line 7?
3. Changing the temperature changes the loss. Why doesn't it change Recall@1?

<details>
<summary>Show answers</summary>

1. We built `images` and `captions` from the same ordered `sample_ids`, so image `i` and caption `i` are a true pair. If the captions got shuffled, `arange(n)` would point at the wrong captions, and the loss would be rewarding the model for wrong matches.
2. `[32, 32]`. Each image row has 1 correct caption and **31** negatives. Bigger batches mean more negatives for free, which is one reason CLIP trained with really big batches.
3. Recall@1 only looks at the `argmax` of each row or column. Multiplying everything by one positive number keeps the order the same, so the argmax doesn't move. The loss looks at the size of the gaps, and those do change.

</details>

## Try it yourself

### TODO 1: Write your own search query

Look back at the contact sheet, then describe one of the images in your own words. The cell below runs your text through lines 2 and 4 of Figure 3 and scores it against our four images. It runs as-is with a generic default, so swap in your own description.

Try a really specific description, then a vague one. Which gives a clearer winner?

In [ ]:
# TODO: describe one of the four images in your own words
my_query = "a photo of a dog"
my_inputs = processor(text=[my_query], return_tensors="pt", padding=True).to(device)
with torch.inference_mode():
    my_features = model.text_model(**my_inputs).pooler_output  # line 2
    my_vector = F.normalize(model.text_projection(my_features), dim=-1)  # line 4

my_scores = my_vector @ image_embeddings.T  # one row: my query vs. every image
print("cosine with each image:", [round(score, 3) for score in my_scores[0].tolist()])
print("best match: I" + str(my_scores.argmax().item() + 1))

### TODO 2: Break the alignment on purpose

The whole objective leans on one assumption: image `i` goes with caption `i`. What if that's wrong? The cell below shuffles the caption vectors (by default it swaps the first two) and recomputes the loss with the *same* diagonal targets. Before you run it, guess: will the loss go up or down?

In [ ]:
# TODO: try other orders, like reversing the whole list
my_order = [1, 0] + list(range(2, SAMPLE_COUNT))  # swap the first two captions
shuffled_logits = logit_scale * (image_embeddings @ text_embeddings[my_order].T)
shuffled_loss = (F.cross_entropy(shuffled_logits, targets) + F.cross_entropy(shuffled_logits.T, targets)) / 2
print("caption order:", my_order)
print("original loss:", round(symmetric_loss.item(), 4), "| shuffled loss:", round(shuffled_loss.item(), 4))

<details>
<summary>Hint</summary>

If CLIP likes the true pairs, the shuffled loss should usually come out higher: now the targets point at the wrong captions. During training, mislabeled pairs like this push the model in the wrong direction. That's why web-scraped datasets with noisy captions are tricky.

</details>

### TODO 3: One caption, or two?

In Section 5 we joined captions A and B. But was that better than just using caption B on its own? This cell searches with a single caption per image. Use `1` (caption B) to compare with Section 5, or `0` to reproduce the Section 4 caption queries.

In [ ]:
# TODO: 0 = the caption from our batch, 1 = the second caption (stick to 0 or 1; every image has at least two)
my_caption_index = 1
my_texts = [captions_by_id[cocoid][my_caption_index] for cocoid in sample_ids]
my_text_inputs = processor(text=my_texts, return_tensors="pt", padding=True).to(device)
with torch.inference_mode():
    my_text_vectors = F.normalize(model.text_projection(model.text_model(**my_text_inputs).pooler_output), dim=-1)

my_matches = (my_text_vectors @ image_embeddings.T).argmax(dim=1)
print("single caption → image index:", my_matches.tolist())
print("single-caption R@1:", my_matches.eq(targets).float().mean().item(), "| composite R@1:", composite_matches.eq(targets).float().mean().item())

With only four images, there's not much room for differences, so a tie is pretty likely. Look at the actual scores too (`my_text_vectors @ image_embeddings.T`): does the joined query win by a bigger margin?

### TODO 4 (optional, slower): Grow the batch

Back in Section 1.2, change `SAMPLE_COUNT` from 4 to 8 and do **Run All**. (It reloads the dataset and the model, so give it a minute.) Now every image has 7 negatives instead of 3. Does the diagonal still stand out? Does Recall@1 drop? With more candidates, there are more ways to be wrong.

> **Heads up:** The prose in this notebook talks about "four" and "4 × 4" all over the place. With a bigger batch, read those as `n` and `n × n`.

---

**Up next, Notebook 04:** we've only *inspected* a pretrained CLIP so far. Next we build a tiny MiniCLIP of our own and train it from scratch with this exact symmetric loss, then watch the diagonal show up as it learns.